# Etapa 2 – CNN com PyTorch
Classificação de cães e gatos usando Rede Neural Convolucional implementada com PyTorch e TorchVision.
**Sem** modelos pré-treinados — arquitetura definida do zero.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
import time

## Configurações e Hiperparâmetros

In [ ]:
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE   = 32
EPOCHS       = 20
LEARNING_RATE = 0.001
DATASET_DIR  = 'datasets'   # ajuste se necessário
NUM_CLASSES  = 2            # cat=0, dog=1

print(f'Dispositivo: {DEVICE}')

## Pré-processamento e Data Augmentation

Aplicamos **três técnicas de grupos diferentes**, conforme exigido:
1. **Geométrico** – `RandomHorizontalFlip`
2. **Rotação** – `RandomRotation(10°)`
3. **Cor** – `ColorJitter` (brilho, contraste, saturação)

Imagens redimensionadas para 224×224 em RGB (3 canais).

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    # 1. Geométrico
    transforms.RandomHorizontalFlip(),
    # 2. Rotação
    transforms.RandomRotation(10),
    # 3. Cor
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

## Carregamento do Dataset

In [ ]:
import os

train_dataset = datasets.ImageFolder(os.path.join(DATASET_DIR, 'train'),      transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(DATASET_DIR, 'validation'), transform=val_transform)
test_dataset  = datasets.ImageFolder(os.path.join(DATASET_DIR, 'test'),       transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Classes: {train_dataset.classes}')   # ['cat', 'dog']
print(f'Train: {len(train_dataset)} imagens')
print(f'Val:   {len(val_dataset)} imagens')
print(f'Test:  {len(test_dataset)} imagens')

## Definição da Arquitetura CNN

Arquitetura customizada com:
- **3 blocos convolucionais** (Conv → BN → ReLU → MaxPool)
- **1 camada Dropout** para regularização
- **2 camadas fully connected** na saída

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes=2):
        super(CNN, self).__init__()

        # Bloco 1: 3 → 32 canais, 224x224 → 112x112
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)   # 224 → 112
        )

        # Bloco 2: 32 → 64 canais, 112x112 → 56x56
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)   # 112 → 56
        )

        # Bloco 3: 64 → 128 canais, 56x56 → 28x28
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)   # 56 → 28
        )

        # Classificador: 128 * 28 * 28 → 256 → num_classes
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x


model     = CNN(num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Resumo simples
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nParâmetros treináveis: {total_params:,}')

## Treinamento

In [ ]:
train_losses = []
val_losses   = []

start = time.time()

for epoch in range(EPOCHS):
    # ========================
    # TREINO
    # ========================
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # ========================
    # VALIDAÇÃO
    # ========================
    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            outputs  = model(images)
            loss     = criterion(outputs, labels)
            val_running_loss += loss.item()

    val_loss = val_running_loss / len(val_loader)
    val_losses.append(val_loss)

    print(f'Época {epoch+1:2d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')

end = time.time()
training_time = end - start
print(f'\nTempo de treinamento: {training_time:.2f}s')

## Avaliação no Conjunto de Teste

In [ ]:
model.eval()
correct    = 0
total      = 0
all_preds  = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs   = model(images)
        _, predicted = torch.max(outputs, 1)

        total   += labels.size(0)
        correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = 100 * correct / total
print(f'Acurácia no teste: {accuracy:.2f}%')
print(f'Tempo de treinamento: {training_time:.2f}s')

## Matriz de Confusão

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
TN, FP, FN, TP = cm.ravel()

print('Matriz de Confusão:')
print(f'              Predito Gato  Predito Cachorro')
print(f'Real Gato         {TN:5d}          {FP:5d}')
print(f'Real Cachorro     {FN:5d}          {TP:5d}')

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f'\nPrecisão:  {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-Score:  {f1:.4f}')

## Gráfico de Loss (Treino vs Validação)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses,   label='Validation Loss')
plt.title('CNN PyTorch – Curva de Loss')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()